In [1]:
! pip install langchain langchain-community langchain-core chromadb sentence-transformers pypdf

In [2]:
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import OllamaEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_community.chat_models import ChatOllama
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_community.document_loaders import PyPDFLoader
import ollama
import uuid

In [3]:
# 1. Load the Target Document (from PDF)
pdf_path = r"C:\Users\acer\Downloads\Global HR Leave Policy Expansion.pdf" 
print(f"Loading document from {pdf_path}...")
loader = PyPDFLoader(pdf_path)
docs = loader.load()

Loading document from C:\Users\acer\Downloads\Global HR Leave Policy Expansion.pdf...


In [4]:
print(docs)

[Document(metadata={'producer': 'Skia/PDF m148 Google Docs Renderer', 'creator': 'PyPDF', 'creationdate': '', 'title': 'Global HR Leave Policy Expansion', 'source': 'C:\\Users\\acer\\Downloads\\Global HR Leave Policy Expansion.pdf', 'total_pages': 16, 'page': 0, 'page_label': '1'}, page_content='ACME  Corporation  Global  Human  \nResources\n \nPolicy:\n \nComprehensive\n \nAbsence\n \nand\n \nLeave\n \nFramework\n \nDocument  ID:  HR-BEN-402-RevC  \nSubject:  Comprehensive  Parental,  Medical,  Civic,  and  Statutory  Leave  Provisions  \nEffective  Date:  January  1,  2026  \n1.0  Purpose  and  Strategic  Scope  \nThe  purpose  of  this  document  is  to  delineate  the  overarching  framework  governing  \nremuneration,\n \njob\n \nprotection,\n \nand\n \nauthorized\n \nabsenteeism\n \nfor\n \neligible\n \npersonnel\n \nexperiencing\n \na\n \nQualifying\n \nFamily\n \nEvent.\n \nThis\n \npolicy\n \nsupersedes\n \nall\n \nprior\n \ndirectives\n \nregarding\n \nmaternity,\n \npaternit

In [5]:
# 2. Chunking with LangChain
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=2000, 
    chunk_overlap=500
)

splits = text_splitter.split_documents(docs)

print(f"Successfully split the PDF into {len(splits)} chunks!")

Successfully split the PDF into 32 chunks!


In [6]:
splits[1]

Document(metadata={'producer': 'Skia/PDF m148 Google Docs Renderer', 'creator': 'PyPDF', 'creationdate': '', 'title': 'Global HR Leave Policy Expansion', 'source': 'C:\\Users\\acer\\Downloads\\Global HR Leave Policy Expansion.pdf', 'total_pages': 16, 'page': 0, 'page_label': '1'}, page_content='it\n \nfunctions\n \nas\n \na\n \nsingle\n \nsource\n \nof\n \ntruth\n \nthat\n \nprevents\n \ninconsistent,\n \nad-hoc\n \nmanagerial\n \ndecisions\n \nwhich\n \nfrequently\n \nprecipitate\n \ndiscrimination\n \nclaims\n \nand\n \namplify\n \nlegal\n \nrisk\n \nexposure.\n1\n By  comprehensively  mapping  eligibility  \ncriteria,\n \napplication\n \nworkflows,\n \nduration\n \nmaximums,\n \nand\n \nreturn-to-work\n \nprocedures\n \nfor\n \nall\n \nleave\n \ntypes—ranging\n \nfrom\n \nstandard\n \nvacation\n \nand\n \ncatastrophic\n \nillness\n \nto\n \ncivic\n \nduty\n \nand\n \nprofessional\n \nsabbaticals—this\n \nframework\n \nmitigates\n \noperational\n \nambiguity.\n2\n Clear  guidelines  

In [7]:
splits[2]

Document(metadata={'producer': 'Skia/PDF m148 Google Docs Renderer', 'creator': 'PyPDF', 'creationdate': '', 'title': 'Global HR Leave Policy Expansion', 'source': 'C:\\Users\\acer\\Downloads\\Global HR Leave Policy Expansion.pdf', 'total_pages': 16, 'page': 1, 'page_label': '2'}, page_content='Terminology  Formal  Administrative  Definition  and  Contextual  Application  \nQualifying  Family  Event  The  biological  parturition  of  a  child,  the  legal  placement  of  a  minor  for  adoption,  or  the  formal  instatement  of  a  foster  care  arrangement.  \nPrimary  Caregiver  The  personnel  member  who  assumes  the  principal  physiological  burden  of  parturition  and/or  the  majority  responsibility  for  the  immediate  care  and  nurturing  of  the  dependent  during  the  immediate  post-placement  or  post-partum  period.  \nSecondary  Caregiver  The  personnel  member  who  shares  a  legal  or  biological  relationship  to  the  dependent  but  does  not  assume  the 

In [8]:
# CREATING EMBEDDING AND VECTOR STORE


# 3. Embedding and Storing in Chroma Vector Store via Ollama
print("Embedding chunks into Chroma Vector Store...")
embeddings = OllamaEmbeddings(model="qwen3-embedding:latest")

# Generate a unique ID for every single chunk
unique_ids = [str(uuid.uuid4()) for _ in range(len(splits))]

# Pass the unique IDs into the database so it can't overwrite them
vectorstore = Chroma.from_documents(
    documents=splits, 
    embedding=embeddings,
    ids=unique_ids 
)


stored_data = vectorstore.get()
print(f"Total chunks successfully stored: {len(stored_data['ids'])}")

Embedding chunks into Chroma Vector Store...


C:\Users\acer\AppData\Local\Temp\ipykernel_6532\2374579669.py:6: LangChainDeprecationWarning: The class `OllamaEmbeddings` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the :class:`~langchain-ollama package and should be used instead. To use it run `pip install -U :class:`~langchain-ollama` and import as `from :class:`~langchain_ollama import OllamaEmbeddings``.
  embeddings = OllamaEmbeddings(model="qwen3-embedding:latest")


Total chunks successfully stored: 32


In [9]:
# Fetch everything, making sure to ask for embeddings and documents
data = vectorstore.get(include=["embeddings", "documents"])

data

{'ids': ['6317f31e-6b6c-4b21-b124-dfc0e50640b4',
  '0e007d87-6f0e-4d06-abd9-19982b222f6c',
  '53e43810-a854-410e-8548-99e1044ded09',
  '1b619f6b-5f56-4e25-90c2-1c8d67935fe7',
  '71bcd879-2066-4be1-8a3e-6876645310a9',
  'fb247465-38c3-413b-877d-fd15602896a3',
  '5c5cdcb0-6a2d-42f1-a732-e5ab18a00c19',
  '6630ef4e-2b06-44b0-b0b4-b6ae738c4191',
  '5ee96107-8449-4595-887c-818933405fa7',
  '61389987-3218-4c61-9f7e-ca1214c440e6',
  '383e41e6-11ad-4aa6-917b-e69776defca7',
  '3c2a78f1-b431-4344-b26c-9239fc595c29',
  '37af7732-ec08-4799-a874-e2304e0b27c7',
  'cc287375-3275-4f08-a7db-c3729c83a174',
  'd9f582e4-070e-435f-a585-17b97f5675d3',
  '08ff432c-88f8-49aa-9593-a92fe1ef6599',
  '461a1c3d-8800-4c77-b26b-af7ed5cf7367',
  '460a6555-2cb4-41a7-9e7d-83511eb93f65',
  '9f24d0db-390b-4d9b-90e2-6608314571da',
  'b5989abe-186e-40e6-af29-6ff66d257c6a',
  '73ee31f5-0c2a-4bf2-887a-23863ca37518',
  '0e10989a-c5b2-4ce5-acd2-bbd4416ae691',
  '4c6d0624-ad56-4967-8135-5848eeab85be',
  '4906923b-8250-4286-be1c-

In [10]:
# 1. The casual user question
user_query = "Do I get paid leaves as a mother if I have a baby, how much time off do the dad gets?"
found_chunks = vectorstore.similarity_search(user_query, k=3)

In [11]:
found_chunks

[Document(metadata={'creator': 'PyPDF', 'page_label': '3', 'creationdate': '', 'producer': 'Skia/PDF m148 Google Docs Renderer', 'title': 'Global HR Leave Policy Expansion', 'page': 2, 'total_pages': 16, 'source': 'C:\\Users\\acer\\Downloads\\Global HR Leave Policy Expansion.pdf'}, page_content="spouses,  parents,  and  children).\n5  \nTime  Off  In  Lieu  (TOIL)  Compensatory  leave  granted  to  personnel  to  offset  extra  hours  worked  beyond  their  standard  contracted  schedule,  provided  in  the  form  of  paid  absence  rather  than  financial  overtime  remuneration.\n4  \nLoss  of  Pay  (LOP)  Authorized  or  unauthorized  absenteeism  occurring  when  an  employee  has  exhausted  all  adequate  leave  balances  and  privileges,  resulting  in  a  proportional  financial  deduction  from  their  base  salary.\n8  \nGarden  Leave  An  administrative  period  during  an  employee's  notice  of  resignation  or  termination  wherein  the  individual  remains  on  the  acti

In [12]:
retrieved_context = "\n\n".join([chunk.page_content for chunk in found_chunks])

In [13]:
retrieved_context

'spouses,  parents,  and  children).\n5  \nTime  Off  In  Lieu  (TOIL)  Compensatory  leave  granted  to  personnel  to  offset  extra  hours  worked  beyond  their  standard  contracted  schedule,  provided  in  the  form  of  paid  absence  rather  than  financial  overtime  remuneration.\n4  \nLoss  of  Pay  (LOP)  Authorized  or  unauthorized  absenteeism  occurring  when  an  employee  has  exhausted  all  adequate  leave  balances  and  privileges,  resulting  in  a  proportional  financial  deduction  from  their  base  salary.\n8  \nGarden  Leave  An  administrative  period  during  an  employee\'s  notice  of  resignation  or  termination  wherein  the  individual  remains  on  the  active  payroll  but  is  explicitly  prohibited  from  reporting  to  the  physical  workplace  or  accessing  corporate  infrastructure.\n4  \nProfessional  Rejuvenation  The  formal  strategic  objective  of  sabbatical  leave,  designed  to  prevent  executive  burnout,  stimulate  intellectual

In [14]:
final_prompt = f"""
<role>
You are an expert, highly accurate Corporate Human Resources Assistant. Your primary responsibility is to provide employees with clear, factual, and 
policy-compliant answers. You are professional, objective, and strictly adhere to the provided company documentation.
</role>

<instruction>
Your task is to answer the user's query by strictly adhering to the following rules:

1. STRICT GROUNDING: You must formulate your answer using ONLY the information provided within the <context> block below. 
2. NO OUTSIDE KNOWLEDGE: Do not rely on your pre-existing knowledge, general HR best practices, or external laws. If the answer is not explicitly stated or directly supported by the provided text, it does not exist for the purpose of this response.
3. HANDLING MISSING INFORMATION: If the provided context does not contain the necessary information to fully or partially answer the user's query, you must not guess, assume, or infer. You must reply strictly with: "I do not have the information required to answer that question based on the provided context."
4. TONE AND STYLE: Maintain a polite, professional, and helpful tone. Avoid using HR jargon unless it is defined within the context.
5. FORMATTING: Provide a concise, direct answer. Where applicable, use bullet points to break down complex rules, conditions, or step-by-step procedures for readability.
</instruction>

<context>
{retrieved_context}
</context>

<query>
{user_query}
</query>
"""

response= ollama.generate(
    model="llama3",
    prompt=final_prompt)

print(response["response"])


Based on the provided context, as a mother, you are entitled to a continuous period of authorized absenteeism not exceeding twelve (12) standard work weeks for a Qualifying Family Event, which includes the biological parturition of a child.

As per the terminology formal administrative definition and contextual application, a Primary Caregiver is the personnel member who assumes the principal physiological burden of parturition and/or the majority responsibility for the immediate care and nurturing of the dependent during the immediate post-placement or post-partum period. You are considered a Primary Caregiver as you have given birth to a child.

According to the context, there is no specific mention of paid leave or compensation duration for fathers in this scenario. However, it does specify that organizations that fail to provide equitable secondary caregiver provisions routinely experience higher rates of unmanaged absenteeism and delayed project deliverables, implying that seconda

### Prompt Re-writing

In [15]:
import ollama

# 1. The original casual question
user_query = "Do i get paid leave for babe as a mother, and how much time off do dads get?"
print("Generating Hypothetical Document via Ollama...\n")

hyde_prompt = f"""
<system_role>
You are an advanced Query Rewriter for a dense semantic vector search engine. Your exclusive function is to translate informal user queries into highly technical, keyword-dense corporate HR governance questions.
</system_role>

<task>
Translate the <user_query> into a single, highly concentrated, formal interrogative sentence (a question) that utilizes exact corporate governance terminology.
</task>

<constraints>
1. Output Format: The output MUST be a single cohesive text block. Begin with the broad, high-level policy question, followed immediately by the hypothetical policy clause. 
2. Tone and Function: Do NOT answer the user's specific question. Instead, generate the exact administrative text that a formal HR document would use to dictate the policy.
3. Lexical Mapping (CRITICAL): Translate colloquial user scenarios into the strict corporate administrative definitions. You must draw from the following verified lexicon where contextually appropriate:
   - Family/Birth: "Qualifying Family Event", "Primary Caregiver", "Secondary Caregiver", "Supplemental Bonding Remuneration"
   - PTO/Time Off: "Accrual Bank", "Authorized Absenteeism", "Concurrent Leave"
   - Overtime/Pay: "Time Off In Lieu (TOIL)", "Loss of Pay (LOP)", "Remuneration"
   - Death/Grief: "Consanguinity"
   - Resignation/Termination: "Garden Leave"
   - Career Break: "Professional Rejuvenation"
4. Anti-Stuffing Rule: DO NOT mix unrelated concepts to increase keyword density. ONLY utilize the lexicon terms that directly map to the core intent of the user's specific scenario. Prioritize semantic accuracy over sheer keyword volume.
5. Absolute Output Restriction: NO PREAMBLE. NO EXPLANATIONS. NO LABELS (e.g., do not write "Question:" or "Policy:"). The very first word of your output MUST be the start of the abstracted question.
</constraints>

<user_query>
{user_query}
</user_query>

OUTPUT:
"""
response= ollama.generate(
    model="llama3",
    prompt=hyde_prompt)

print(response["response"])

hypothetical_prompt=response["response"]

Generating Hypothetical Document via Ollama...

What entitlements to Supplemental Bonding Remuneration does an Eligible Employee receive upon occurrence of a Qualifying Family Event involving the birth of a child, and what Authorized Absenteeism provisions are afforded to both Primary Caregivers and Secondary Caregivers during this period, taking into account any applicable Time Off In Lieu (TOIL) or Loss of Pay (LOP)?


In [16]:
# We use the fake, formal document to search the vector database instead of the short query
hyde_found_chunks = vectorstore.similarity_search(hypothetical_prompt, k=3)

retrieved_context = "\n\n".join([chunk.page_content for chunk in hyde_found_chunks])

In [17]:
retrieved_context

"spouses,  parents,  and  children).\n5  \nTime  Off  In  Lieu  (TOIL)  Compensatory  leave  granted  to  personnel  to  offset  extra  hours  worked  beyond  their  standard  contracted  schedule,  provided  in  the  form  of  paid  absence  rather  than  financial  overtime  remuneration.\n4  \nLoss  of  Pay  (LOP)  Authorized  or  unauthorized  absenteeism  occurring  when  an  employee  has  exhausted  all  adequate  leave  balances  and  privileges,  resulting  in  a  proportional  financial  deduction  from  their  base  salary.\n8  \nGarden  Leave  An  administrative  period  during  an  employee's  notice  of  resignation  or  termination  wherein  the  individual  remains  on  the  active  payroll  but  is  explicitly  prohibited  from  reporting  to  the  physical  workplace  or  accessing  corporate  infrastructure.\n4  \nProfessional  Rejuvenation  The  formal  strategic  objective  of  sabbatical  leave,  designed  to  prevent  executive  burnout,  stimulate  intellectual 

In [18]:
final_prompt = f"""
<role>
You are an expert, highly accurate Corporate Human Resources Assistant. Your primary responsibility is to provide employees with clear, factual, and policy-compliant answers. You are professional, objective, and strictly adhere to the provided company documentation.
</role>

<instruction>
Your task is to answer the user's query by strictly adhering to the following rules:

1. STRICT GROUNDING: You must formulate your answer using ONLY the information provided within the <context> block below. 
2. NO OUTSIDE KNOWLEDGE: Do not rely on your pre-existing knowledge, general HR best practices, or external laws. If the answer is not explicitly stated or directly supported by the provided text, it does not exist for the purpose of this response.
3. HANDLING MISSING INFORMATION: If the provided context does not contain the necessary information to fully or partially answer the user's query, you must not guess, assume, or infer. You must reply strictly with: "I do not have the information required to answer that question based on the provided context."
4. TONE AND STYLE: Maintain a polite, professional, and helpful tone. Avoid using HR jargon unless it is defined within the context.
5. FORMATTING: Provide a concise, direct answer. Where applicable, use bullet points to break down complex rules, conditions, or step-by-step procedures for readability.
</instruction>

<context>
{retrieved_context}
</context>

<query>
{user_query}
</query>
"""

response= ollama.generate(
    model="llama3",
    prompt=final_prompt)

print(response["response"])


Based on the provided context, I can answer your query.

As a mother (Primary Caregiver), you are entitled to a continuous period of authorized absenteeism not exceeding twelve standard work weeks. Additionally, during the initial physiological recovery phase (typically six to eight weeks post-parturition, contingent upon medical certification), remuneration shall be administered via the corporation's Short-Term Disability insurance apparatus, yielding income replacement at 100% of your standard base salary.

As for fathers (Secondary Caregivers), they are granted a distinct allocation of compensated absenteeism to support family integration and assist the Primary Caregiver. Secondary Caregivers are authorized to utilize up to four weeks of fully compensated continuous or intermittent leave within twelve calendar months immediately following the Qualifying Family Event.

Please note that unused allocations for Secondary Caregivers are subject to forfeiture and not eligible for monetary

## STEP BACK PROMPTING

In [23]:
user_query= "I had to work 12 extra hours last weekend to finish the big software deployment. How many full days off can I get , if each day shift is of 9 hours."

advanced_retrieval_prompt = f"""
<system_role>
You are an expert AI Query Abstractor and Advanced Query Rewriter for a corporate Human Resources dense semantic vector search engine. Your sole function is to perform "Step-Back Prompting" by translating informal, situational user queries into highly technical, keyword-dense corporate HR governance questions.
</system_role>

<task>
Analyze the user's specific, situational question and translate it into a single, highly concentrated, formal interrogative sentence (a question) that utilizes exact corporate governance terminology. You must strip away all personal narrative, emotional context, and specific constraints, focusing entirely on the overarching corporate governance principle that dictates the answer.
</task>

<rules_and_constraints>
1. DO NOT answer the user's specific question. Instead, generate the exact administrative text that a formal HR document would use to dictate the policy.
2. DO NOT include specific situational details (e.g., "my grandfather", "next Tuesday", "12 extra hours", "National Guard") in your output.
3. Lexical Mapping (CRITICAL): Translate colloquial user scenarios into strict corporate administrative definitions. You must draw from the following verified lexicon where contextually appropriate:
   - Family/Birth: "Qualifying Family Event", "Primary Caregiver", "Secondary Caregiver", "Supplemental Bonding Remuneration"
   - PTO/Time Off: "Accrual Bank", "Authorized Absenteeism", "Concurrent Leave"
   - Overtime/Pay: "Time Off In Lieu (TOIL)", "Loss of Pay (LOP)", "Remuneration"
   - Death/Grief: "Consanguinity"
   - Resignation/Termination: "Garden Leave"
   - Career Break: "Professional Rejuvenation"
4. Anti-Stuffing Rule: DO NOT mix unrelated concepts to increase keyword density. ONLY utilize the lexicon terms that directly map to the core intent of the user's specific scenario. Prioritize semantic accuracy over sheer keyword volume.
5. Output Format: The output MUST be a single cohesive text block. Begin with the broad, high-level policy question, followed immediately by the hypothetical policy clause.
6. Absolute Output Restriction: NO PREAMBLE. NO EXPLANATIONS. NO LABELS (e.g., do not write "Question:", "Step-Back Question:", or "Policy:"). The very first word of your output MUST be the start of the abstracted question.
</rules_and_constraints>

<examples>
Original Question: My grandfather passed away yesterday and I need to travel out of state for the funeral. How many days off can I take to handle this?
Abstracted Query: What is the corporate policy regarding the duration and authorization of compensated bereavement leave for the death of a second-degree relative of consanguinity?
e
Original Question: I've been with the company for seven years and I'm totally burned out. Can I take a paid month off to go study advanced analytics and come back refreshed?
Abstracted Query: What are the eligibility requirements, parameters, and remuneration guidelines for extended sabbatical leave or professional rejuvenation for tenured personnel?

Original Question: I'm in the National Guard and just got called up for a 3-week emergency deployment. Will my company pay get cut while I'm gone?
Abstracted Query: What is the remuneration and authorized absenteeism framework for personnel activated for emergency uniformed military service or contingency deployments?

Original Question: My ex is stalking me and I need to go to court on Wednesday to get a restraining order. Will I get in trouble or lose pay for missing work?
Abstracted Query: What job protections and authorized absenteeism provisions exist for personnel who require time off to secure emergency judicial relief related to victim status or domestic violence?
</examples>

<input>
Original Question: {user_query}
</input>
"""

response= ollama.generate(
    model="llama3",
    prompt=advanced_retrieval_prompt)

print(response["response"])

hypothetical_prompt=response["response"]

What is the corporate policy regarding authorized overtime compensation and accrual bank management for personnel who have worked extended shifts or irregular schedules, and what are the procedures for requesting compensatory time off in lieu (TOIL) or loss of pay (LOP) for accrued hours?


In [24]:
# We use the fake, formal document to search the vector database instead of the short query
hyde_found_chunks = vectorstore.similarity_search(hypothetical_prompt, k=3)
retrieved_context = "\n\n".join([chunk.page_content for chunk in hyde_found_chunks])

In [25]:
retrieved_context

'primary\n \ndefensive\n \nlegal\n \napparatus\n \nto\n \nenforce\n \nnon-competition\n \nclauses\n \nand\n \nsystematically\n \nprevent\n \nthe\n \nimmediate\n \nhemorrhaging\n \nof\n \nproprietary\n \ndata\n \nto\n \nmarket\n \nrivals\n \nwhile\n \nthe\n \nemployee\'s\n \ninstitutional\n \nknowledge\n \ndegrades.\n4  \n12.0  Compensatory  Leave  and  Time  Off  In  Lieu  (TOIL)  \nTo  manage  periods  of  acute  operational  intensity,  project  deployment  surges,  or  emergency  \nweekend\n \ndemands\n \nwithout\n \ntriggering\n \nsystemic\n \npayroll\n \ninflation,\n \nthe\n \norganization\n \nemploys\n \nCompensatory\n \nLeave,\n \ncommonly\n \nreferred\n \nto\n \nglobally\n \nas\n \nTime\n \nOff\n \nin\n \nLieu\n \n(TOIL).\n4  \nWhen  personnel  work  beyond  their  standard  contracted  schedule,  they  may  accrue  additional  \npaid\n \ntime\n \noff\n \nrather\n \nthan\n \nreceiving\n \na\n \ndirect\n \nfinancial\n \novertime\n \ndisbursement.\n4\n In  the  United  \nStates,\

In [26]:
final_prompt = f"""
<role>
You are an expert, highly accurate Corporate Human Resources Assistant. Your primary responsibility is to provide employees with clear, factual, and policy-compliant answers. You are professional, objective, and strictly adhere to the provided company documentation.
</role>

<instruction>
Your task is to answer the user's query by strictly adhering to the following rules:

1. STRICT GROUNDING: You must formulate your answer using ONLY the information provided within the <context> block below. 
2. NO OUTSIDE KNOWLEDGE: Do not rely on your pre-existing knowledge, general HR best practices, or external laws. If the answer is not explicitly stated or directly supported by the provided text, it does not exist for the purpose of this response.
3. HANDLING MISSING INFORMATION: If the provided context does not contain the necessary information to fully or partially answer the user's query, you must not guess, assume, or infer. You must reply strictly with: "I do not have the information required to answer that question based on the provided context."
4. TONE AND STYLE: Maintain a polite, professional, and helpful tone. Avoid using HR jargon unless it is defined within the context.
5. FORMATTING: Provide a concise, direct answer. Where applicable, use bullet points to break down complex rules, conditions, or step-by-step procedures for readability.
</instruction>

<context>
{retrieved_context}
</context>

<query>
{user_query}
</query>
"""

response= ollama.generate(
    model="gpt-oss:120b-cloud",
    prompt=final_prompt)

print(response["response"])

- You accrue TOIL at a **time‑and‑a‑half** rate.  
  12 extra hours × 1.5 = **18 hours** of compensatory leave.  

- With a standard shift of **9 hours**, 18 hours of TOIL provides:  
  18 ÷ 9 = **2 full days** off.  
